In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

device_torch = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device_torch}")





Using device: cpu


In [4]:
import pandas as pd
import numpy as np

df = pd.read_pickle('research\models\impute_result\df_complete_clean.pkl')
df = df.sort_values('timestamp').reset_index(drop=True)

In [5]:

df = pd.read_pickle('research\models\impute_result\df_complete_clean.pkl')
df = df.sort_values('timestamp').reset_index(drop=True)
df['sin_COG'] = np.sin(np.radians(df['COG']))
df['cos_COG'] = np.cos(np.radians(df['COG']))
df['jitter_log'] = np.log1p(df['jitter'])

SEQ_LEN = 20
GAP_THRESHOLD = 60
HIDDEN_SIZE = 256       
NUM_LAYERS = 2         
DROPOUT = 0.20          
BATCH_SIZE = 512        
LR = 0.0003           
EPOCHS = 100            
PATIENCE = 15  


In [6]:
column_name=df.columns.to_list()

In [7]:
BASE_INPUTS = [
    'hour', 'day_of_week',
    'device_pc1', 'device_pc2', 'device_pc3', 'device_pc4',
    'direction_uplink',
    'measured_qos_delay',
    'measurement', 'operator'
]

ALL_TARGETS = [
    'Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG', 'Altitude',
    'precipIntensity', 'precipProbability', 'temperature', 'humidity', 'windSpeed',
    'Traffic Jam Factor', 'Traffic Distance',
    'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
    'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
    'PCell_Downlink_frequency', 'PCell_Band_Indicator',
    'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz',
    'PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs', 'PCell_Downlink_TB_Size',
    'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High',
    'PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size', 'PCell_Uplink_Tx_Power_(dBm)',
    'datarate', 'jitter_log', 'Pos in Ref Round', 'target_datarate',
    'ping_ms'
]

SNAP_RULES = {
    'PCell_Downlink_frequency': [125.0, 475.0, 1300.0, 1801.0, 2850.0, 3050.0, 3749.0, 9460.0],
    'PCell_freq_MHz': [700.0, 900.0, 1800.0, 2000.0, 2100.0, 2600.0],
    'PCell_Band_Indicator': [1.0, 3.0, 7.0, 8.0, 28.0],
    'PCell_Downlink_bandwidth_MHz': [5.0, 10.0, 15.0, 20.0],
    'PCell_Uplink_bandwidth_MHz': [5.0, 10.0, 15.0, 20.0],
    'PCell_Downlink_Average_MCS': list(range(0, 30)),
}

def snap_to_nearest(values, valid_set):
    valid_arr = np.array(valid_set)
    result = np.empty_like(values)
    for i, v in enumerate(values):
        result[i] = valid_arr[np.argmin(np.abs(valid_arr - v))]
    return result



In [9]:

device_cols_list = ['device_pc1', 'device_pc2', 'device_pc3', 'device_pc4']
all_segments = []
for d_col in device_cols_list:
    sub = df[df[d_col] == 1].sort_values('timestamp').reset_index(drop=True)
    gaps = sub['timestamp'].diff().dt.total_seconds()
    break_indices = gaps[gaps > GAP_THRESHOLD].index.tolist()
    starts = [0] + break_indices
    ends = break_indices + [len(sub)]
    for s, e in zip(starts, ends):
        seg = sub.iloc[s:e].reset_index(drop=True)
        if len(seg) >= SEQ_LEN:
            all_segments.append(seg)

print(f"Segments: {len(all_segments)}, Total rows: {sum(len(s) for s in all_segments)}")

all_timestamps = df['timestamp'].sort_values()
n = len(all_timestamps)
t_train_end = all_timestamps.iloc[int(n * 0.70)]
t_val_end = all_timestamps.iloc[int(n * 0.85)]

def get_split(ts):
    ts = pd.Timestamp(ts, tz='Europe/Berlin')
    if ts <= t_train_end:
        return 'train'
    elif ts <= t_val_end:
        return 'val'
    else:
        return 'test'



Segments: 64, Total rows: 204942


In [ ]:

def create_sequences_direct(segments, base_cols, auto_cols, target_cols, split='train'):
    """
    Input at each timestep = base_cols + auto_cols (all targets as autoregressive)
    Target: all target values at the next timestep (timestep after window)
    """
    X_list, y_list = [], []
    for seg_df in segments:
        seg_timestamps = seg_df['timestamp'].values
        seg_base = seg_df[base_cols].values
        seg_auto = seg_df[auto_cols].values
        seg_targets = seg_df[target_cols].values

        for i in range(SEQ_LEN, len(seg_df)):
            if get_split(pd.Timestamp(seg_timestamps[i])) != split:
                continue

            base_seq = seg_base[i - SEQ_LEN:i]
            auto_seq = seg_auto[i - SEQ_LEN:i].copy()
            auto_seq[-1, :] = 0.0

            x = np.concatenate([base_seq, auto_seq], axis=1)
            y = seg_targets[i]

            X_list.append(x)
            y_list.append(y)

    if not X_list:
        return None, None
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

print("\nCreating sequences...")
t0 = time.time()
X_train, y_train = create_sequences_direct(all_segments, BASE_INPUTS, ALL_TARGETS, ALL_TARGETS, 'train')
X_val, y_val = create_sequences_direct(all_segments, BASE_INPUTS, ALL_TARGETS, ALL_TARGETS, 'val')
X_test, y_test = create_sequences_direct(all_segments, BASE_INPUTS, ALL_TARGETS, ALL_TARGETS, 'test')
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)} ({time.time()-t0:.1f}s)")



Creating sequences...
Train: 157599, Val: 26910, Test: 19153 (92.3s)


In [11]:

n_feat = X_train.shape[2]
input_scaler = StandardScaler()
X_train = input_scaler.fit_transform(X_train.reshape(-1, n_feat)).reshape(-1, SEQ_LEN, n_feat)
X_val = input_scaler.transform(X_val.reshape(-1, n_feat)).reshape(-1, SEQ_LEN, n_feat)
X_test = input_scaler.transform(X_test.reshape(-1, n_feat)).reshape(-1, SEQ_LEN, n_feat)

target_scaler = StandardScaler()
y_train_s = target_scaler.fit_transform(y_train)
y_val_s = target_scaler.transform(y_val)

print(f"Input dim: {n_feat}, Output dim: {len(ALL_TARGETS)}")



Input dim: 47, Output dim: 37


In [ ]:

class LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim,
                            num_layers=num_layers,
                            dropout=dropout if num_layers > 1 else 0,
                            batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])



In [ ]:

def train_model(model, model_name, X_train, y_train_s, X_val, y_val_s, X_test, y_test):
    print(f"\n{'='*60}")
    print(f"  TRAINING: {model_name}")
    print(f"{'='*60}")

    model = model.to(device_torch)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    criterion = nn.MSELoss()

    train_loader = DataLoader(
        torch.utils.data.TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train_s)),
        batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(
        torch.utils.data.TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val_s)),
        batch_size=BATCH_SIZE, shuffle=False)

    best_val_loss = float('inf')
    best_state = None
    pat = 0
    t0 = time.time()

    for epoch in range(EPOCHS):
        model.train()
        t_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device_torch), yb.to(device_torch)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss += loss.item() * len(xb)
        t_loss /= len(X_train)

        model.eval()
        v_loss = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device_torch), yb.to(device_torch)
                v_loss += criterion(model(xb), yb).item() * len(xb)
        v_loss /= len(X_val)
        scheduler.step(v_loss)

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1

        if (epoch + 1) % 5 == 0 or pat == 0:
            print(f"  Epoch {epoch+1:3d} | Train: {t_loss:.6f} | Val: {v_loss:.6f} | "
                  f"LR: {optimizer.param_groups[0]['lr']:.6f} | Pat: {pat}")

        if pat >= PATIENCE:
            print(f"  Early stop at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)
    model.eval()
    train_time = time.time() - t0

    # evaluation
    with torch.no_grad():
        pred_s = model(torch.FloatTensor(X_test).to(device_torch)).cpu().numpy()
    pred = target_scaler.inverse_transform(pred_s)

    tf_results = {}
    for i, t in enumerate(ALL_TARGETS):
        p = pred[:, i].copy()
        if t in SNAP_RULES:
            p = snap_to_nearest(p, SNAP_RULES[t])
        rmse = np.sqrt(mean_squared_error(y_test[:, i], p))
        mae = mean_absolute_error(y_test[:, i], p)
        tf_results[t] = {'rmse_tf': rmse, 'mae_tf': mae}

    print(f"\n  Training time: {train_time:.1f}s ({train_time/60:.1f} min)")
    return model, tf_results, train_time


n_input = n_feat
n_output = len(ALL_TARGETS)


# LSTM
lstm_model = LSTM(n_input, HIDDEN_SIZE, n_output, NUM_LAYERS, DROPOUT)
lstm_model, lstm_results, lstm_time = train_model(
    lstm_model, "LSTM", X_train, y_train_s, X_val, y_val_s, X_test, y_test)





  TRAINING: Direct LSTM
  Epoch   1 | Train: 0.401411 | Val: 0.272494 | LR: 0.000300 | Pat: 0
  Epoch   2 | Train: 0.193774 | Val: 0.219477 | LR: 0.000300 | Pat: 0
  Epoch   3 | Train: 0.175298 | Val: 0.209825 | LR: 0.000300 | Pat: 0
  Epoch   4 | Train: 0.166553 | Val: 0.203593 | LR: 0.000300 | Pat: 0
  Epoch   5 | Train: 0.161346 | Val: 0.206663 | LR: 0.000300 | Pat: 1
  Epoch   8 | Train: 0.152773 | Val: 0.200357 | LR: 0.000300 | Pat: 0
  Epoch  10 | Train: 0.150126 | Val: 0.204064 | LR: 0.000300 | Pat: 2
  Epoch  12 | Train: 0.147864 | Val: 0.200322 | LR: 0.000300 | Pat: 0
  Epoch  13 | Train: 0.147076 | Val: 0.199627 | LR: 0.000300 | Pat: 0
  Epoch  15 | Train: 0.145688 | Val: 0.205067 | LR: 0.000300 | Pat: 2
  Epoch  20 | Train: 0.141641 | Val: 0.203799 | LR: 0.000150 | Pat: 7
  Epoch  25 | Train: 0.140394 | Val: 0.206244 | LR: 0.000075 | Pat: 12
  Early stop at epoch 28

  Training time: 4095.2s (68.3 min)


In [10]:
print(f"OUTPUT: LSTM ")
print(f"{'Feature':40s} | {'RMSE':>12s} | {'MAE':>12s}")
print("─" * 70)

for col in ALL_TARGETS:
    l_rmse = lstm_results.get(col, {}).get('rmse_tf', 0)
    l_mae  = lstm_results.get(col, {}).get('mae_tf', 0)

    display = col
    if col == 'jitter_log':
        display = 'jitter'

    print(f"{display:40s} | {l_rmse:12.4f} | {l_mae:12.4f}")


OUTPUT: LSTM 
Feature                                  |         RMSE |          MAE
──────────────────────────────────────────────────────────────────────
Latitude                                 |       0.0059 |       0.0043
Longitude                                |       0.0186 |       0.0152
speed_kmh                                |       0.7093 |       0.5394
sin_COG                                  |       0.4143 |       0.3448
cos_COG                                  |       0.2259 |       0.1722
Altitude                                 |       4.5018 |       3.8347
precipIntensity                          |       0.1154 |       0.0921
precipProbability                        |       0.2432 |       0.2268
temperature                              |       0.3871 |       0.3253
humidity                                 |       0.0359 |       0.0337
windSpeed                                |       0.3088 |       0.2723
Traffic Jam Factor                       |       1.8184 |      

In [ ]:
torch.save(lstm_model.state_dict(), 'research/generation_output/lstm_model.pt')
pickle.dump({'lstm_tf': lstm_results},
            open('research/generation_output/lstm_multioutput_results.pkl', 'wb'))
pickle.dump({'input_scaler': input_scaler, 'target_scaler': target_scaler},
            open('research/generation_output/lstm_multioutput_scalers.pkl', 'wb'))

print("\nSaved all models and results.")
print("Done!")


Saved all models and results.
Done!
